# PlugIR baseline vs. S1 · Hits@10

The two methods use the same 200 dialogue records and 50,000-image index. They stop when the target first ranks in the top 10, or after 10 answers. Round 0 is the search before any questions. A hit counts for that round and every later round.

S1 searches with the original description and the latest reconstruction and combines the two rankings with reciprocal rank fusion. Each unique query contributes \(1/(60+\mathrm{rank})\) per image. The resulting ranking also supplies the candidate pool for the same PlugIR k-means and representative selection used by the baseline. Both arms use the same 500-to-50 pool schedule and pinned BLIP and Qwen checkpoints.

The progress and plotting cells can read the included saved results on a CPU. The run cell requires an Apple Silicon Mac and the two environments described in the README.

In [ ]:
from pathlib import Path
import json
import sys
import subprocess

# Start Jupyter in the directory containing this notebook, or set PACKAGE explicitly.
PACKAGE = Path.cwd().resolve()
RESULTS = PACKAGE / "results"
REPORT = PACKAGE / "source" / "report_hits.py"

# Change these paths if your checkout or MLX environment lives elsewhere.
PROJECT = next(
    (p for p in (PACKAGE, *PACKAGE.parents) if (p / "ChatIR-Cornell2026").is_dir()),
    PACKAGE.parent,
)
REPO = next(
    (p for p in (PACKAGE, *PACKAGE.parents) if (p / ".git").exists()),
    PROJECT / "ChatIR-Cornell2026",
)
BLIP_PYTHON = REPO / ".venv" / "bin" / "python"
MLX_ENV = PROJECT / "tmp" / "agentir-mlx-vlm-env"
QWEN_REVISION = "8b2b98c00a6b4d291155e4890773ca8f769aee53"
SNAPSHOT = (REPO / "cache/huggingface/hub/models--mlx-community--Qwen3.5-9B-4bit"
            / "snapshots" / QWEN_REVISION)
ARMS = ["vanilla", "s1_original_current"]
SAMPLE = json.loads((PACKAGE / "sample.json").read_text())
DIALOGUE_COUNT = len(SAMPLE["tasks"])
print(f"{DIALOGUE_COUNT} dialogues per method, seed {SAMPLE['seed']}")

## Check saved progress

A file is written for every completed dialogue. The paired count includes IDs with a saved result for both methods.

In [ ]:
def saved_ids(arm):
    return {int(path.stem) for path in (RESULTS / arm / "dialogues").glob("*.json")}

for arm in ARMS:
    print(f"{arm}: {len(saved_ids(arm))}/{DIALOGUE_COUNT}")
print(f"Matched: {len(set.intersection(*(saved_ids(arm) for arm in ARMS)))}/{DIALOGUE_COUNT}")

## Run or resume on Apple Silicon

Run this cell only when you want to start inference. If a process is already writing to this results folder, use the progress cell. Set `RESULTS` to a new directory before starting an independent replication. The saved settings and source hashes are checked on resume.

In [ ]:
from shutil import copy2
from tempfile import TemporaryDirectory

for dependency in (BLIP_PYTHON, MLX_ENV / "bin/python", SNAPSHOT / "config.json"):
    if not dependency.exists():
        raise FileNotFoundError(f"{dependency} is missing. See README.md for setup.")

# Stage the recorded source in the layout used by the original runner.
with TemporaryDirectory(prefix="agentir-s1-") as stage:
    workspace = Path(stage)
    bundle_dir = workspace / "deliverables/full_dialogue_colab"
    source_dir = bundle_dir / "source"
    source_dir.mkdir(parents=True)
    (workspace / "ChatIR-Cornell2026").symlink_to(REPO, target_is_directory=True)
    (workspace / "tmp").mkdir()
    (workspace / "tmp/agentir-mlx-vlm-env").symlink_to(MLX_ENV, target_is_directory=True)
    for source in (PACKAGE / "source").glob("*.py"):
        copy2(source, source_dir / source.name)
    archive = "agentir_full_dialogue_bundle_v2.zip"
    (bundle_dir / archive).symlink_to(PACKAGE / archive)
    subprocess.run(
        ["caffeinate", "-i", str(BLIP_PYTHON), "-u", str(source_dir / "local_hits.py"),
         "--snapshot", str(SNAPSHOT), "--output", str(RESULTS), "--limit", str(DIALOGUE_COUNT),
         "--arms", *ARMS],
        check=True,
    )

## Plot the matched comparison

The default requires all 200 pairs. Set `ALLOW_PARTIAL = True` to inspect an interim curve labeled with its smaller sample size.

In [ ]:
from IPython.display import Image, display

ALLOW_PARTIAL = False
command = [sys.executable, str(REPORT), str(RESULTS)]
if ALLOW_PARTIAL:
    command.append("--allow-partial")
figures = subprocess.check_output(command, text=True).strip()
display(Image(filename=str(Path(figures) / "hits_at_10.png")))